In [1]:
import json
import yaml
from mstrio.connection import Connection

from mstr_robotics.mstr_classes import mstr_global
from mstr_robotics._helper import msic
from mstr_robotics.redis_db import  redis_bi_analysis,redis_mstr_json
from mstr_robotics._connectors import mstr_api
from mstr_robotics.prepare_AI_data import export_mstr_md


i_msic=msic()
i_mstr_global=mstr_global()
i_mstr_api=mstr_api()
i_redis_mstr_json=redis_mstr_json()
i_export_mstr_md=export_mstr_md()


with open('..\\config\\mstr_redis_y.yml', 'r') as openfile:
    mstr_redis_y = yaml.safe_load(openfile)


In [2]:
redis_con_d=mstr_redis_y["redis_env_d"]["redis_dev"]
project_prefix=mstr_redis_y["project_prefix"]
project_id=redis_con_d["project_id"]
prefix_map=mstr_redis_y["prefix_map"]
searches_used_in_prp_d_l=mstr_redis_y["searches_used_in_prp_d_l"]


## Connect to MSTR & Redis

In [3]:
with open('..\\config\\user_d.json', 'r') as openfile:
    user_d = json.load(openfile)
conn_params =  user_d["conn_params"]
conn = Connection(**conn_params)


i_redis_bi_analysis = redis_bi_analysis( 
    host=redis_con_d["host"],
    port=redis_con_d["port"],
    password=redis_con_d["password"],
    username=redis_con_d["username"],
    decode_responses=redis_con_d["decode_responses"]
)


Connection to Strategy One Intelligence Server has been established.


## Save objects to redis

In [ ]:
load_type_fg='daily_search_update'
load_type_fg='full_load'
load_type_fg='shortcut_folder'
load_type_fg='single_objects'

### Daily Search Update

In [7]:
if load_type_fg=="daily_search_update":
    err_d_l=[]
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(project_id)
        search_result = i_mstr_api.run_mstr_search(conn=conn,
                            search_id="96648F2B492150A6AA27DDB3744E32B4")
    
        if search_result["totalItems"] > 0:
            all_obj_d_l=search_result["result"]
    
            err_d_l.append(i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                            ,conn=conn
                                                            ,prefix_map=prefix_map
                                                            , all_obj_d_l=all_obj_d_l
                                                            , env_prefix=env_prefix)
                            )

### Full load

In [ ]:

#load_type_fg="full_load"
if load_type_fg=="full_load":
    # Initialize Redis connection
    env_prefix="mstr_test"
    i_redis_bi_analysis.emergency_flush_db(confirm_phrase='FLUSH_ALL_DATA')
    
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(pre)
        all_obj_d_l=i_export_mstr_md.read_out_prj_by_type(conn)
    
        err_d_l=i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis
                                                        ,conn=conn
                                                        ,prefix_map=prefix_map
                                                        , all_obj_d_l=all_obj_d_l
                                                        , env_prefix=env_prefix)
    
        i_redis_mstr_json.run_searches_redis(conn=conn
                                            ,i_redis_bi_analysis=i_redis_bi_analysis
                                            ,prefix_map=prefix_map
                                            ,env_prefix=env_prefix
                                            ,searches_used_in_prp_d_l=searches_used_in_prp_d_l)
      

### Shortcut Folder

In [ ]:
if load_type_fg=="shortcut_folder":
    folder_id="B70AF11248C00F7D4339079310F8B3F1"
    conn.select_project(project_id)
    all_obj_d_l=i_mstr_global.get_obj_from_sh_fold(conn,folder_id=folder_id)

### Single Objects

In [ ]:
if load_type_fg=="single_objects":
    err_d_l=[]
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(project_id)
        all_obj_d_l=[]
        all_obj_d_l.append({"id":"D63BE643469DE5784DD83C8F95B8BFE6","type":"3","subtype":"776"})
        i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                    ,conn=conn
                                                    ,prefix_map=prefix_map
                                                    , all_obj_d_l=all_obj_d_l
                                                    , env_prefix=env_prefix)

Project selected in Connection object:
Project object named: 'MicroStrategy Tutorial' with ID: 'B7CA92F04B9FAE8D941C3E9B7E0CD754'
Uploaded object definition for D63BE643469DE5784DD83C8F95B8BFE6
Uploaded object definition for D63BE643469DE5784DD83C8F95B8BFE6
